# Gemma 4 — Tool Calling

## Imports

In [1]:
import json
import re
from pprint import pprint

import torch
import transformers
from transformers.utils import get_json_schema

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("MPS (Apple Silicon GPU) available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

torch: 2.13.0
transformers: 5.14.1
MPS (Apple Silicon GPU) available: True
CUDA available: False


## Load Model and Processor

In [2]:
MODEL_ID = "google/gemma-4-E2B-it"

processor = transformers.AutoProcessor.from_pretrained(MODEL_ID)
model = transformers.AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

print(f"Architecture: {model.config.architectures}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Architecture: ['Gemma4ForConditionalGeneration']
Parameters: 5,104,297,504
Device: mps:0


## Define Tools

In [3]:
def get_weather(location: str):
    """
    Get the current weather for a location.

    Args:
        location: The city to get the weather for, e.g. "San Francisco"
    """
    return {"location": location, "temperature": 68, "unit": "F", "conditions": "sunny"}


def get_stock_price(ticker: str):
    """
    Get the latest closing price for a stock ticker.

    Args:
        ticker: The stock ticker symbol, e.g. "AAPL"
    """
    return {"ticker": ticker, "price": 294.38, "currency": "USD"}


def convert_currency(amount: float, from_currency: str, to_currency: str):
    """
    Convert an amount of money from one currency to another.

    Args:
        amount: The amount of money to convert
        from_currency: The currency code to convert from, e.g. "USD"
        to_currency: The currency code to convert to, e.g. "EUR"
    """
    rate = 0.88
    return {"amount": round(amount * rate, 2), "currency": to_currency, "rate": rate}


TOOL_REGISTRY = {
    "get_weather": get_weather,
    "get_stock_price": get_stock_price,
    "convert_currency": convert_currency,
}

In [4]:
tools = [get_json_schema(fn) for fn in TOOL_REGISTRY.values()]
pprint(tools, sort_dicts=False, width=120)

[{'type': 'function',
  'function': {'name': 'get_weather',
               'description': 'Get the current weather for a location.',
               'parameters': {'type': 'object',
                              'properties': {'location': {'type': 'string',
                                                          'description': 'The city to get the weather for, e.g. "San '
                                                                         'Francisco"'}},
                              'required': ['location']}}},
 {'type': 'function',
  'function': {'name': 'get_stock_price',
               'description': 'Get the latest closing price for a stock ticker.',
               'parameters': {'type': 'object',
                              'properties': {'ticker': {'type': 'string',
                                                        'description': 'The stock ticker symbol, e.g. "AAPL"'}},
                              'required': ['ticker']}}},
 {'type': 'function',
  'function': {'

Gemma 4 emits tool calls as `<|tool_call>call:name{key:value}<tool_call|>`, where string values are delimited by `<|"|>` instead of quotes. Those are special tokens, so the response has to be decoded with them intact and then converted back to JSON.

In [5]:
TOOL_CALL_PATTERN = re.compile(r"<\|tool_call>call:(\w+)\{(.*?)\}<tool_call\|>", re.DOTALL)
UNQUOTED_KEY = re.compile(r"([{,])\s*([A-Za-z_]\w*)\s*:")


def decode_response(outputs, input_len):
    text = processor.decode(outputs[0][input_len:], skip_special_tokens=False).strip()
    return text.removesuffix("<turn|>").strip()


def parse_tool_calls(text):
    tool_calls = []
    for name, raw_arguments in TOOL_CALL_PATTERN.findall(text):
        arguments = UNQUOTED_KEY.sub(r'\1"\2":', "{" + raw_arguments.replace('<|"|>', '"') + "}")
        tool_calls.append({
            "type": "function",
            "function": {"name": name, "arguments": json.loads(arguments)},
        })
    return tool_calls

## Single Tool Call

In [6]:
messages = [
    {"role": "user", "content": "What's the weather in San Francisco?"},
]

chat = processor.apply_chat_template(
    messages,
    tools=tools,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(chat)

<bos><|turn>system
<|tool>declaration:get_weather{description:<|"|>Get the current weather for a location.<|"|>,parameters:{properties:{location:{description:<|"|>The city to get the weather for, e.g. "San Francisco"<|"|>,type:<|"|>STRING<|"|>}},required:[<|"|>location<|"|>],type:<|"|>OBJECT<|"|>}}<tool|><|tool>declaration:get_stock_price{description:<|"|>Get the latest closing price for a stock ticker.<|"|>,parameters:{properties:{ticker:{description:<|"|>The stock ticker symbol, e.g. "AAPL"<|"|>,type:<|"|>STRING<|"|>}},required:[<|"|>ticker<|"|>],type:<|"|>OBJECT<|"|>}}<tool|><|tool>declaration:convert_currency{description:<|"|>Convert an amount of money from one currency to another.<|"|>,parameters:{properties:{amount:{description:<|"|>The amount of money to convert<|"|>,type:<|"|>NUMBER<|"|>},from_currency:{description:<|"|>The currency code to convert from, e.g. "USD"<|"|>,type:<|"|>STRING<|"|>},to_currency:{description:<|"|>The currency code to convert to, e.g. "EUR"<|"|>,type:<|

In [7]:
inputs = processor(text=chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = decode_response(outputs, input_len)

print(response)

<|tool_call>call:get_weather{location:<|"|>San Francisco<|"|>}<tool_call|><|tool_response>


In [8]:
tool_calls = parse_tool_calls(response)
pprint(tool_calls, sort_dicts=False, width=120)

[{'type': 'function', 'function': {'name': 'get_weather', 'arguments': {'location': 'San Francisco'}}}]


In [9]:
messages.append({"role": "assistant", "tool_calls": tool_calls})

for call in tool_calls:
    name, arguments = call["function"]["name"], call["function"]["arguments"]
    result = TOOL_REGISTRY[name](**arguments)
    print(f"{name}({arguments}) -> {result}")
    messages.append({"role": "tool", "name": name, "content": json.dumps(result)})

pprint(messages, sort_dicts=False, width=120)

get_weather({'location': 'San Francisco'}) -> {'location': 'San Francisco', 'temperature': 68, 'unit': 'F', 'conditions': 'sunny'}
[{'role': 'user', 'content': "What's the weather in San Francisco?"},
 {'role': 'assistant',
  'tool_calls': [{'type': 'function',
                  'function': {'name': 'get_weather', 'arguments': {'location': 'San Francisco'}}}]},
 {'role': 'tool',
  'name': 'get_weather',
  'content': '{"location": "San Francisco", "temperature": 68, "unit": "F", "conditions": "sunny"}'}]


In [10]:
chat = processor.apply_chat_template(
    messages,
    tools=tools,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(chat)

<bos><|turn>system
<|tool>declaration:get_weather{description:<|"|>Get the current weather for a location.<|"|>,parameters:{properties:{location:{description:<|"|>The city to get the weather for, e.g. "San Francisco"<|"|>,type:<|"|>STRING<|"|>}},required:[<|"|>location<|"|>],type:<|"|>OBJECT<|"|>}}<tool|><|tool>declaration:get_stock_price{description:<|"|>Get the latest closing price for a stock ticker.<|"|>,parameters:{properties:{ticker:{description:<|"|>The stock ticker symbol, e.g. "AAPL"<|"|>,type:<|"|>STRING<|"|>}},required:[<|"|>ticker<|"|>],type:<|"|>OBJECT<|"|>}}<tool|><|tool>declaration:convert_currency{description:<|"|>Convert an amount of money from one currency to another.<|"|>,parameters:{properties:{amount:{description:<|"|>The amount of money to convert<|"|>,type:<|"|>NUMBER<|"|>},from_currency:{description:<|"|>The currency code to convert from, e.g. "USD"<|"|>,type:<|"|>STRING<|"|>},to_currency:{description:<|"|>The currency code to convert to, e.g. "EUR"<|"|>,type:<|

In [11]:
inputs = processor(text=chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = decode_response(outputs, input_len)

print(response)

The weather in San Francisco is 68°F and sunny.


## Multiple Tool Calls

In [12]:
messages = [
    {"role": "user", "content": "What's the weather in San Francisco and the stock price of AAPL?"},
]

inputs = processor.apply_chat_template(
    messages,
    tools=tools,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = decode_response(outputs, input_len)

print(response)

<|tool_call>call:get_weather{location:<|"|>San Francisco<|"|>}<tool_call|><|tool_call>call:get_stock_price{ticker:<|"|>AAPL<|"|>}<tool_call|><eos>


In [13]:
tool_calls = parse_tool_calls(response)
pprint(tool_calls, sort_dicts=False, width=120)

[{'type': 'function', 'function': {'name': 'get_weather', 'arguments': {'location': 'San Francisco'}}},
 {'type': 'function', 'function': {'name': 'get_stock_price', 'arguments': {'ticker': 'AAPL'}}}]


In [14]:
messages.append({"role": "assistant", "tool_calls": tool_calls})

for call in tool_calls:
    name, arguments = call["function"]["name"], call["function"]["arguments"]
    result = TOOL_REGISTRY[name](**arguments)
    print(f"{name}({arguments}) -> {result}")
    messages.append({"role": "tool", "name": name, "content": json.dumps(result)})

pprint(messages, sort_dicts=False, width=120)

get_weather({'location': 'San Francisco'}) -> {'location': 'San Francisco', 'temperature': 68, 'unit': 'F', 'conditions': 'sunny'}
get_stock_price({'ticker': 'AAPL'}) -> {'ticker': 'AAPL', 'price': 294.38, 'currency': 'USD'}
[{'role': 'user', 'content': "What's the weather in San Francisco and the stock price of AAPL?"},
 {'role': 'assistant',
  'tool_calls': [{'type': 'function', 'function': {'name': 'get_weather', 'arguments': {'location': 'San Francisco'}}},
                 {'type': 'function', 'function': {'name': 'get_stock_price', 'arguments': {'ticker': 'AAPL'}}}]},
 {'role': 'tool',
  'name': 'get_weather',
  'content': '{"location": "San Francisco", "temperature": 68, "unit": "F", "conditions": "sunny"}'},
 {'role': 'tool', 'name': 'get_stock_price', 'content': '{"ticker": "AAPL", "price": 294.38, "currency": "USD"}'}]


In [15]:
inputs = processor.apply_chat_template(
    messages,
    tools=tools,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = decode_response(outputs, input_len)

print(response)

The weather in San Francisco is 68°F and sunny. The stock price for AAPL is $294.38 USD.


## Sequential Tool Calls

Some requests need the result of one tool before the next can be called. The system prompt keeps the model from guessing an argument it cannot know yet.

In [16]:
messages = [
    {
        "role": "system",
        "content": (
            "You must call tools one at a time and wait for each result before deciding "
            "the next step. Never guess a tool argument that depends on a previous tool result."
        ),
    },
    {"role": "user", "content": "What is AAPL's stock price converted to EUR?"},
]

print(f"[user]\n{messages[-1]['content']}\n")

while True:
    inputs = processor.apply_chat_template(
        messages,
        tools=tools,
        tokenize=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    outputs = model.generate(**inputs, max_new_tokens=128)
    response = decode_response(outputs, input_len)

    tool_calls = parse_tool_calls(response)
    if not tool_calls:
        break

    messages.append({"role": "assistant", "tool_calls": tool_calls})
    for call in tool_calls:
        name, arguments = call["function"]["name"], call["function"]["arguments"]
        result = TOOL_REGISTRY[name](**arguments)
        print(f"[tool call]\n{name}({arguments})\n")
        print(f"[tool response]\n{result}\n")
        messages.append({"role": "tool", "name": name, "content": json.dumps(result)})

print("[assistant]")
print(response)

[user]
What is AAPL's stock price converted to EUR?

[tool call]
get_stock_price({'ticker': 'AAPL'})

[tool response]
{'ticker': 'AAPL', 'price': 294.38, 'currency': 'USD'}

[tool call]
convert_currency({'amount': 294.38, 'from_currency': 'USD', 'to_currency': 'EUR'})

[tool response]
{'amount': 259.05, 'currency': 'EUR', 'rate': 0.88}

[assistant]
The stock price for AAPL is $294.38 USD, which is approximately 259.05 EUR.


In [17]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'system',
  'content': 'You must call tools one at a time and wait for each result before deciding the next step. Never guess a '
             'tool argument that depends on a previous tool result.'},
 {'role': 'user', 'content': "What is AAPL's stock price converted to EUR?"},
 {'role': 'assistant',
  'tool_calls': [{'type': 'function', 'function': {'name': 'get_stock_price', 'arguments': {'ticker': 'AAPL'}}}]},
 {'role': 'tool', 'name': 'get_stock_price', 'content': '{"ticker": "AAPL", "price": 294.38, "currency": "USD"}'},
 {'role': 'assistant',
  'tool_calls': [{'type': 'function',
                  'function': {'name': 'convert_currency',
                               'arguments': {'amount': 294.38, 'from_currency': 'USD', 'to_currency': 'EUR'}}}]},
 {'role': 'tool', 'name': 'convert_currency', 'content': '{"amount": 259.05, "currency": "EUR", "rate": 0.88}'},
 {'role': 'assistant', 'content': 'The stock price for AAPL is $294.38 USD, which is approximately 259.05 EUR.'

In [18]:
conversation = processor.apply_chat_template(
    messages,
    tools=tools,
    add_generation_prompt=False,
)

print(conversation)

<bos><|turn>system
You must call tools one at a time and wait for each result before deciding the next step. Never guess a tool argument that depends on a previous tool result.<|tool>declaration:get_weather{description:<|"|>Get the current weather for a location.<|"|>,parameters:{properties:{location:{description:<|"|>The city to get the weather for, e.g. "San Francisco"<|"|>,type:<|"|>STRING<|"|>}},required:[<|"|>location<|"|>],type:<|"|>OBJECT<|"|>}}<tool|><|tool>declaration:get_stock_price{description:<|"|>Get the latest closing price for a stock ticker.<|"|>,parameters:{properties:{ticker:{description:<|"|>The stock ticker symbol, e.g. "AAPL"<|"|>,type:<|"|>STRING<|"|>}},required:[<|"|>ticker<|"|>],type:<|"|>OBJECT<|"|>}}<tool|><|tool>declaration:convert_currency{description:<|"|>Convert an amount of money from one currency to another.<|"|>,parameters:{properties:{amount:{description:<|"|>The amount of money to convert<|"|>,type:<|"|>NUMBER<|"|>},from_currency:{description:<|"|>The